# 4. Compare logged model results

Airflow waits for both training notebooks before starting comparison. Only these two successful runs compete.

## Execution parameters
Airflow injects these values into the tagged parameter cell. For manual execution, keep the defaults and run the five notebooks in order.

In [ ]:
# Notebook inputs.
# Airflow supplies the paths and run ID; manual execution uses these defaults.
project_root = ""
run_dir = ""
workflow_run_id = ""
experiment_name = "iris-pipeline-comparison"
registered_model_name = "iris-pipeline-classifier"

## Locate files and connect to MLflow
SQLite holds experiment and registry records. This run folder holds data and small result files shared between notebook tasks.

In [ ]:
import json
from pathlib import Path

import mlflow
import pandas as pd
from mlflow import MlflowClient

# Find the project folder.
# Looking for requirements.txt lets this notebook start from a subfolder too.
if project_root:
    project_path = Path(project_root).resolve()
else:
    project_path = Path.cwd().resolve()
while not (project_path / "requirements.txt").is_file():
    if project_path == project_path.parent:
        raise RuntimeError("Open the notebook from inside the workshop project.")
    project_path = project_path.parent

# Choose the output folder.
# Separate Airflow run folders prevent one execution from overwriting another.
if run_dir:
    run_path = Path(run_dir)
else:
    run_path = project_path / "outputs" / "manual"
run_path.mkdir(parents=True, exist_ok=True)

# Connect to MLflow.
# One database keeps both models, their runs, and registered versions together.
tracking_uri = "sqlite:///" + (project_path / "mlflow.db").as_posix()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
client = MlflowClient()

print("Run folder:", run_path)

## Read this execution
Reject stale result files before comparing. Manual users should run the first notebook and both training notebooks for each new comparison.

In [ ]:
split = json.loads((run_path / "split.json").read_text(encoding="utf-8"))
logistic_result = json.loads(
    (run_path / "logistic_result.json").read_text(encoding="utf-8"),
)
forest_result = json.loads(
    (run_path / "forest_result.json").read_text(encoding="utf-8"),
)
for result in [logistic_result, forest_result]:
    assert result["execution_id"] == split["execution_id"], "Run both models for the current split."

# Read the completed training runs.
# Their metric dictionaries contain the final values stored by MLflow.
logistic_run = client.get_run(logistic_result["run_id"])
forest_run = client.get_run(forest_result["run_id"])
assert logistic_run.info.status == "FINISHED"
assert forest_run.info.status == "FINISHED"
# Build the comparison rows.
# copy keeps the original handoff intact; update adds the model's saved metrics.
logistic_row = logistic_result.copy()
logistic_row.update(logistic_run.data.metrics)

forest_row = forest_result.copy()
forest_row.update(forest_run.data.metrics)

comparison = pd.DataFrame([logistic_row, forest_row])
comparison[[
    "model_type", "train_accuracy", "test_accuracy",
    "train_log_loss", "test_log_loss", "training_seconds",
]]

## Select and save the winner
Highest accuracy wins. A tie chooses Logistic Regression because it is the first row. This tiny holdout demonstrates the lifecycle; it is not a production evaluation strategy.

In [ ]:
winner_index = comparison["test_accuracy"].idxmax()
winner = comparison.loc[winner_index].to_dict()
(run_path / "winner.json").write_text(json.dumps(winner, indent=2), encoding="utf-8")
print("Winner:", winner["model_type"])
print("Accuracy:", winner["accuracy"])
print("Source run:", winner["run_id"])